# Aralin 13 - Memorya ng Ahente gamit ang Cognee Knowledge Graphs


## Setup

Ipinapakita ng notebook na ito kung paano bumuo ng isang matalinong **coding assistant** na may persistenteng memorya gamit ang mga knowledge graph ng [**Cognee**](https://www.cognee.ai/) at ang **Microsoft Agent Framework** (MAF).

Ginagawang structured, queryable knowledge graph na suportado ng vector embeddings ng Cognee ang hindi nakaayos na teksto — nagbibigay ito sa iyong ahente ng mayamang long-term memory na may kamalayan sa relasyon.

### Ang Malalaman Mo
1. **Bumuo ng Knowledge Graphs**: I-transform ang mga developer profiles at pinakamahusay na mga kasanayan sa istrukturadong, queryable na kaalaman.
2. **Isama ang Cognee sa MAF**: Gamitin ang mga `@tool` function para payagan ang MAF agent na mag-query sa knowledge graph ng Cognee.
3. **Session-Aware Conversations**: Panatilihin ang konteksto sa iba't ibang tanong sa parehong session.
4. **Long-Term Memory**: Itago ang mahahalagang kaalaman sa paglipas ng mga session at kunin ito sa mga bagong pag-uusap.

### Mga Kinakailangan
- Python 3.9+
- Patakbuhin ang Redis nang lokal (`docker run -d -p 6379:6379 redis`) para sa pamamahala ng session
- Isang LLM API key (hal. OpenAI) — itakda ang `LLM_API_KEY` sa `.env`
- `CACHING=true` sa `.env` (kailangan para sa Cognee sessions)
- Isang Microsoft Foundry project na may deployed chat model
- `AZURE_AI_PROJECT_ENDPOINT` at `AZURE_AI_MODEL_DEPLOYMENT_NAME` sa `.env`
- Na-authenticate sa Azure CLI (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## Mga Uri ng Memorya ng Ahente

Tinutuklas ng notebook na ito ang parehong tatlong uri ng memorya mula sa pangunahing Lesson 13 notebook, ngunit gumagamit ng Cognee bilang long-term memory backend:

| Uri ng Memorya | Mekanismo | Haba ng Buhay |
|---|---|---|
| **Pangunahing Gawain** | `agent.create_session()` (MAF) | Isang pag-uusap |
| **Pangmadaliang Panahon** | Cognee session cache (Redis) | Isang sesyon |
| **Pangmatagalan** | Cognee knowledge graph + vectors | Pangmatagalan |

### Arkitektura ng Memorya ng Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Ihanda ang Imbakan ng Cognee


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Bahagi 1 — Pagbuo ng Knowledge Base

Kumukuha kami ng tatlong uri ng datos upang makabuo ng komprehensibong knowledge base para sa aming coding assistant:

1. **Developer Profile** — personal na kasanayan at teknikal na background
2. **Python Best Practices** — ang Zen ng Python na may mga praktikal na alituntunin
3. **Historical Conversations** — mga nakaraang sesyon ng tanong at sagot sa pagitan ng mga developer at mga AI assistant


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Ipakita ang Knowledge Graph

Kayang magpakita ng interactive na HTML visualization ang Cognee ng mga entidad at relasyon na na-extract nito.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Payamanin ang Memorya gamit ang Memify

Sinusuri ng `memify()` ang knowledge graph at bumubuo ng matatalinong patakaran — tinutukoy ang mga pattern, pinakamahuhusay na kasanayan, at mga ugnayan sa pagitan ng mga konsepto.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Bahagi 2 — MAF Agent gamit ang Cognee Tools

Ngayon ay gagawa tayo ng isang MAF agent na maaaring mag-query sa knowledge graph ng Cognee gamit ang mga function na `@tool`. Pinapahintulutan nito ang agent na gamitin ang buong kapangyarihan ng graph-aware semantic search habang pinananatili ang konteksto ng pag-uusap sa pamamagitan ng mga session.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## Working Memory with Sessions

Ang `AgentSession` (na nilikha gamit ang `agent.create_session()`) ay nagbibigay ng working memory sa loob ng isang session. Maaaring balikan ng ahente ang mga naunang mensahe habang hinihiling din ang long-term knowledge graph ng Cognee.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Bagong Session — Nagpapatuloy ang Pangmatagalang Alaala

Ang pagsisimula ng bagong session ay naglilinis ng working memory, ngunit ang Cognee knowledge graph ay nananatiling magagamit. Maaari ng ahente na kunin ang parehong pangmatagalang kaalaman sa isang ganap na bagong pag-uusap.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Buod

Sa notebook na ito, bumuo ka ng isang coding assistant na pinagsasama ang **working memory ng MAF** (`agent.create_session()`) at **long-term knowledge graph ng Cognee**.

### Ano ang Iyong Natutunan
1. **Pagbuo ng knowledge graph**: Tinatanggap ng Cognee ang unstructured na teksto at bumubuo ng isang graph + vector memory.
2. **Pagpapayaman ng graph gamit ang memify**: Mga hango na katotohanan at mas mayamang mga ugnayan sa ibabaw ng umiiral mong graph.
3. **Integrasyon ng MAF + Cognee**: Ang mga `@tool` na function ay nagpapahintulot sa MAF agent na natural na mag-query ng graph ng Cognee.
4. **Working memory + long-term memory**: Ang `AgentSession` (sa pamamagitan ng `agent.create_session()`) ay nagbibigay ng session context habang ang Cognee ay nagbibigay ng persistent knowledge.
5. **Filtered search gamit ang NodeSets**: Target ang mga partikular na subset ng knowledge graph (hal. mga prinsipyo lamang).

### Mahahalagang Punto
- Ang **Cognee** ay nagbabago ng raw na teksto sa istrakturadong, may kamalayan sa ugnayan na memorya — mas makapangyarihan kaysa payak na vector store.
- Ang mga **`@tool` function** ay nag-uugnay ng mga MAF agent at mga panlabas na knowledge system ng maayos.
- Ang **`AgentSession`** (sa pamamagitan ng `agent.create_session()`) ay naghihiwalay ng per-conversation context mula sa pangmatagalang kaalaman.
- Ang parehong knowledge graph ay nagsisilbi sa maraming session at agent.

### Mga Praktikal na Aplikasyon
- **Developer copilots**: Review ng code, pagsusuri ng insidente, mga katulong sa arkitektura
- **Customer-facing copilots**: Mga ahente ng suporta sa mga dokumento ng produkto, FAQs, at mga tala sa CRM
- **Internal expert copilots**: Mga katulong sa polisiya, legal, o seguridad na nagrereason sa mga alituntunin
- **Pinag-isang mga layer ng data**: Pinagsasama ang istrakturadong at unstructured na data sa isang queryable na graph

### Mga Susunod na Hakbang
- Mag-eksperimento sa temporal awareness sa Cognee
- Magtakda ng OWL ontology para sa kalidad ng graph na partikular sa domain
- Magdagdag ng user feedback loops upang mapabuti ang retrieval sa paglipas ng panahon
- Mag-scale sa multi-agent systems na nagbabahagi ng parehong Cognee memory layer


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
